In [1]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl

In [2]:
DATASET_PATH = Path("../neural_fx_dataset/")
TEST_DI = DATASET_PATH / "DI.wav"
TEST_EFFECT = DATASET_PATH / "tsmini_gain_100.wav"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class _L(nn.LSTM):
    """
    Tweaks to PyTorch LSTM module
    * Up the remembering
    """

    def reset_parameters(self) -> None:
        super().reset_parameters()
        # https://danijar.com/tips-for-training-recurrent-neural-networks/
        # forget += 1
        # ifgo
        value = 2.0
        idx_input = slice(0, self.hidden_size)
        idx_forget = slice(self.hidden_size, 2 * self.hidden_size)
        for layer in range(self.num_layers):
            for input in ("i", "h"):
                # Balance out the scale of the cell w/ a -=1
                getattr(self, f"bias_{input}h_l{layer}").data[idx_input] -= value
                getattr(self, f"bias_{input}h_l{layer}").data[idx_forget] += value
    
class NeuralfxConfig:
    def __init__(self, input_size, hidden_size, num_layers, conv_filters, kernel_size, stride, dropout):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.conv_filters = conv_filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.dropout = dropout

class NeuralfxBaseModel(nn.Module):
    def __init__(self, config):
        super(NeuralfxBaseModel, self).__init__()
        self.config = config

        self.padding = self.calculate_same_padding(config.input_size, config.kernel_size, config.stride)

        self.conv_in = nn.Conv1d(1, config.conv_filters, config.kernel_size, stride=config.stride, padding=self.padding)
        self.activation = nn.ELU()
        self.fc_out = nn.Linear(config.hidden_size, 1)

    def calculate_same_padding(self, input_size, kernel_size, stride):
        out_size = (input_size + stride - 1) // stride
        padding_needed = max(0, (out_size - 1) * stride + kernel_size - input_size)
        return padding_needed // 2

    def forward(self, x):
        x = self.conv_in(x)
        x = self.activation(x)
        x = self.process_sequence(x)
        x = self.fc_out(x)
        x = x.transpose(1, 2)
        return x

    def process_sequence(self, x):
        raise NotImplementedError

class NeuralfxRecurrentModel(NeuralfxBaseModel):
    def __init__(self, config):
        super(NeuralfxRecurrentModel, self).__init__(config)

    def process_sequence(self, x):
        x = self.conv(x)
        x = self.activation_2(x)
        x = x.transpose(1, 2)
        x, _ = self.rnn(x)
        return x

class NeuralfxLSTM(NeuralfxRecurrentModel):
    def __init__(self, config):
        super(NeuralfxLSTM, self).__init__(config)
        self.conv = nn.Conv1d(16, config.conv_filters, config.kernel_size, stride=config.stride, padding=self.padding)
        self.activation_2 = self.activation = nn.ELU()
        self.rnn = _L(
            input_size=config.conv_filters,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            dropout=config.dropout,
            batch_first=True
        )

class NeuralfxGRU(NeuralfxRecurrentModel):
    def __init__(self, config):
        super(NeuralfxGRU, self).__init__(config)
        self.conv = nn.Conv1d(16, config.conv_filters, config.kernel_size, stride=config.stride, padding=self.padding)
        self.activation_2 = self.activation = nn.ELU()
        self.rnn = nn.GRU(
            input_size=config.conv_filters,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            dropout=config.dropout,
            batch_first=True
        )

In [4]:
def pre_emphasis_filter(x, coeff=0.95):
    return torch.cat([x[:, :1], x[:, 1:] - coeff * x[:, :-1]], dim=1)

def ESR(y_pred, y_true):
    y_true, y_pred = pre_emphasis_filter(y_true), pre_emphasis_filter(y_pred)
    return torch.sum(torch.pow(y_true - y_pred, 2)) / (torch.sum(torch.pow(y_true, 2)) + 1e-10)

class NeuralfxLightningModule(pl.LightningModule):
    def __init__(self, model, learning_rate):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('val_loss', loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

In [5]:
class AudioDataset(Dataset):
    def __init__(self, di_file, effected_file, segment_length=32768):
        self.x, self.sr = torchaudio.load(di_file)
        self.y, _ = torchaudio.load(effected_file)
        
        if self.x.shape[1] != self.y.shape[1]:
            raise ValueError(f"Mismatch in audio lengths between DI and effected files")
        
        self.segment_length = segment_length
        self.num_segments = self.x.shape[1] // self.segment_length

    def __len__(self):
        return self.num_segments

    def __getitem__(self, idx):
        start = idx * self.segment_length
        end = start + self.segment_length

        di_segment = self.x[:, start:end]
        effected_segment = self.y[:, start:end]

        return di_segment, effected_segment
    



In [6]:
# Hyperparameters
learning_rate = 0.01
conv1d_filters = 16
conv1d_strides = 12
hidden_units = 36
batch_size = 16
num_epochs = 10
input_size = 150

# File paths
di_file = "../neural_fx_dataset/DI.wav"
effected_file = "../neural_fx_dataset/tsmini_gain_100.wav"

# Create config and model
config = NeuralfxConfig(
    input_size=input_size,
    hidden_size=hidden_units,
    num_layers=2,
    conv_filters=conv1d_filters,
    kernel_size=3,
    stride=conv1d_strides,
    dropout=0.1
)

model = NeuralfxLSTM(config)

# Create dataset and dataloaders
dataset = AudioDataset(di_file, effected_file, segment_length=input_size)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=4)

# Create Lightning module and trainer
Neuralfx_module = NeuralfxLightningModule(model, learning_rate)
trainer = pl.Trainer(max_epochs=num_epochs, accelerator=DEVICE)

# Train the model
trainer.fit(Neuralfx_module, train_loader, val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3050 OEM') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type         | Params
-------------------------------------------
0 | model     | NeuralfxLSTM | 19.3 K
1 | criterion | MSELoss      | 0     
-------------------------------------------
19.3 K    Trainable params
0         Non-trainable params
19.3 K    Total params
0.077     Total estimated model params size (MB)
2024-08-18 12:04:42.216565: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary i

Sanity Checking: 0it [00:00, ?it/s]

/home/ikkjo/anaconda3/envs/neural-fx/lib/python3.11/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([16, 1, 150])) that is different to the input size (torch.Size([16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (16) must match the size of tensor b (150) at non-singleton dimension 2